# MilletSaarthi — Agent 3: Market Data Preparation

**Agent 3 has NO ML model.** It uses:
1. A static list of APMC markets (`apmc_markets.json`) — built in this notebook
2. A live price scraper (also built here) — pulls today's modal prices from agmarknet.gov.in
3. Distance + transport cost math (already in `agents_all.ipynb`)

This notebook produces the JSON file and tests the scraper. Run it once and commit the JSON; the scraper runs at request time.

## 1. Setup

In [ ]:
!pip -q install requests beautifulsoup4 geopy
import json, time, random, requests
from bs4 import BeautifulSoup
from pathlib import Path
from geopy.distance import geodesic

## 2. APMC markets master list

Maharashtra has 300+ APMCs. We pick **20+ major ones** that trade Bajra/Jowar/Ragi.
Lat/lng sourced from district headquarters (good enough for transport cost estimates).

**To expand:** add more dicts to the list below. Source for full list:
https://msamb.com/ApmcDetails (Maharashtra State Agricultural Marketing Board)

In [ ]:
MARKETS = [
    {'name':'Aurangabad','district':'Aurangabad','lat':19.8762,'lng':75.3433,'commission_pct':2.0,'supported':['Bajra','Jowar','Ragi']},
    {'name':'Jalna',     'district':'Jalna',     'lat':19.8347,'lng':75.8816,'commission_pct':1.8,'supported':['Bajra','Jowar']},
    {'name':'Latur',     'district':'Latur',     'lat':18.4088,'lng':76.5604,'commission_pct':1.5,'supported':['Bajra','Jowar']},
    {'name':'Nashik',    'district':'Nashik',    'lat':20.0110,'lng':73.7903,'commission_pct':2.2,'supported':['Bajra','Jowar','Ragi']},
    {'name':'Pune',      'district':'Pune',      'lat':18.5204,'lng':73.8567,'commission_pct':2.0,'supported':['Jowar','Ragi']},
    {'name':'Solapur',   'district':'Solapur',   'lat':17.6599,'lng':75.9064,'commission_pct':1.5,'supported':['Bajra','Jowar']},
    {'name':'Ahmednagar','district':'Ahmednagar','lat':19.0948,'lng':74.7480,'commission_pct':1.8,'supported':['Bajra','Jowar']},
    {'name':'Beed',      'district':'Beed',      'lat':18.9894,'lng':75.7585,'commission_pct':1.5,'supported':['Bajra','Jowar']},
    {'name':'Parbhani',  'district':'Parbhani',  'lat':19.2608,'lng':76.7770,'commission_pct':1.5,'supported':['Bajra','Jowar']},
    {'name':'Nanded',    'district':'Nanded',    'lat':19.1383,'lng':77.3210,'commission_pct':1.8,'supported':['Bajra','Jowar']},
    {'name':'Hingoli',   'district':'Hingoli',   'lat':19.7173,'lng':77.1493,'commission_pct':1.5,'supported':['Jowar']},
    {'name':'Osmanabad', 'district':'Osmanabad', 'lat':18.1860,'lng':76.0419,'commission_pct':1.5,'supported':['Bajra','Jowar']},
    {'name':'Satara',    'district':'Satara',    'lat':17.6805,'lng':74.0183,'commission_pct':2.0,'supported':['Jowar','Ragi']},
    {'name':'Sangli',    'district':'Sangli',    'lat':16.8524,'lng':74.5815,'commission_pct':2.0,'supported':['Jowar','Ragi']},
    {'name':'Kolhapur',  'district':'Kolhapur',  'lat':16.7050,'lng':74.2433,'commission_pct':2.2,'supported':['Jowar','Ragi']},
    {'name':'Akola',     'district':'Akola',     'lat':20.7059,'lng':77.0219,'commission_pct':1.5,'supported':['Jowar']},
    {'name':'Amravati',  'district':'Amravati',  'lat':20.9374,'lng':77.7796,'commission_pct':1.8,'supported':['Jowar']},
    {'name':'Nagpur',    'district':'Nagpur',    'lat':21.1458,'lng':79.0882,'commission_pct':2.0,'supported':['Jowar']},
    {'name':'Wardha',    'district':'Wardha',    'lat':20.7453,'lng':78.6022,'commission_pct':1.8,'supported':['Jowar']},
    {'name':'Buldhana',  'district':'Buldhana',  'lat':20.5292,'lng':76.1842,'commission_pct':1.5,'supported':['Jowar']},
    {'name':'Yavatmal',  'district':'Yavatmal',  'lat':20.3899,'lng':78.1307,'commission_pct':1.5,'supported':['Jowar']},
    {'name':'Dhule',     'district':'Dhule',     'lat':20.9042,'lng':74.7749,'commission_pct':1.8,'supported':['Bajra','Jowar']},
    {'name':'Jalgaon',   'district':'Jalgaon',   'lat':21.0077,'lng':75.5626,'commission_pct':1.8,'supported':['Bajra','Jowar']},
]
print(f'{len(MARKETS)} markets')

## 3. Save to data/apmc_markets.json (Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/MilletSaarthi/data')
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT/'apmc_markets.json','w') as f:
    json.dump(MARKETS, f, indent=2)
print('Saved', OUT/'apmc_markets.json')

## 4. Live price scraper (agmarknet.gov.in)

Agmarknet exposes a public search endpoint that returns an HTML table.
If their site changes, update the selectors. Always wrap in try/except and cache results.

In [ ]:
HEADERS = {'User-Agent':'Mozilla/5.0 (compatible; MilletSaarthi/1.0)'}

def scrape_modal_price(commodity, market, state='Maharashtra'):
    """Returns modal price (₹/quintal) for commodity at market today, or None on failure.
    NOTE: This is a stub. Agmarknet uses ASP.NET viewstate which makes plain HTML POST tricky.
    A robust approach: use the data.gov.in API (free, JSON, requires API key from data.gov.in).
    See https://data.gov.in -> 'Variety-wise Daily Market Prices' -> Get API key.
    """
    try:
        # Example data.gov.in call (requires your API key)
        api_key = 'YOUR_DATA_GOV_IN_KEY'  # put in .env later
        if api_key == 'YOUR_DATA_GOV_IN_KEY':
            return None  # not configured yet
        url = 'https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070'
        params = {
            'api-key': api_key, 'format':'json', 'limit':10,
            'filters[commodity]': commodity,
            'filters[market]':    market,
            'filters[state]':     state,
        }
        r = requests.get(url, params=params, headers=HEADERS, timeout=10)
        records = r.json().get('records', [])
        if not records: return None
        return float(records[0]['modal_price'])
    except Exception as e:
        print(f'scrape failed for {commodity}/{market}: {e}')
        return None

# Test (returns None until API key added)
print(scrape_modal_price('Bajra','Aurangabad'))

## 5. Get a free data.gov.in API key

1. Go to https://data.gov.in and create an account
2. Visit https://data.gov.in/help/how-use-datasets-apis
3. Generate API key (instant, free)
4. Add to `.env` as `DATA_GOV_IN_KEY=...`
5. Replace `YOUR_DATA_GOV_IN_KEY` in the cell above

The relevant resource ID is `9ef84268-d588-465a-a308-a864a43d0070` (Variety-wise Daily Market Prices).

## 6. Distance + net profit demo (Agent 3 logic)

In [ ]:
TRANSPORT_RATE = 20  # ₹/km/quintal
ROAD_FACTOR = 1.3

def best_market(farmer_lat, farmer_lng, millet, qty_quintal, expected_price):
    out = []
    for m in MARKETS:
        if millet not in m['supported']: continue
        live = scrape_modal_price(millet, m['name'])
        price = live if live else expected_price
        d_km = geodesic((farmer_lat,farmer_lng),(m['lat'],m['lng'])).km * ROAD_FACTOR
        gross = price * qty_quintal
        transport = d_km * TRANSPORT_RATE * qty_quintal
        commission = gross * (m['commission_pct']/100)
        net = gross - transport - commission
        out.append({'name':m['name'],'distance_km':round(d_km,1),
                    'price':round(price,2),'net_profit':round(net,2)})
    out.sort(key=lambda x:x['net_profit'], reverse=True)
    return out

# Example: farmer in Aurangabad, 10 quintal Bajra, expected ₹2500/q
ranked = best_market(19.8762, 75.3433, 'Bajra', 10, 2500)
print(json.dumps(ranked[:5], indent=2))